# 🔬 Clasificación de Cáncer de Mama con Machine Learning

**Dataset:** Wisconsin Breast Cancer Dataset (Kaggle)  
**Objetivo:** Clasificar tumores como benignos o malignos a partir de características morfológicas celulares.  
**Pipeline:** EDA → PCA → Modelos de Clasificación → Optimización Bayesiana → Red Neuronal Profunda

---

## 0. Importación de Librerías

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import kagglehub

# Preprocesamiento
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# PCA
from sklearn.decomposition import PCA

# Modelos
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.svm import SVR, SVC
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Métricas y validación
from sklearn.metrics import (
    mean_squared_error, r2_score,
    confusion_matrix, accuracy_score,
    recall_score, precision_score, f1_score,
    roc_auc_score, classification_report
)
from sklearn.model_selection import train_test_split, cross_val_score

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print('✓ Librerías cargadas correctamente')

## 1. Carga y Exploración Inicial de Datos

In [ ]:
# Descargar dataset desde Kaggle
file_path = kagglehub.dataset_download('imtkaggleteam/breast-cancer')
print('Path to dataset files:', file_path)

In [ ]:
# Cargar datos
csv_path = os.path.join(file_path, 'breast-cancer-wisconsin-data_data.csv')
data = pd.read_csv(csv_path)

print('=' * 60)
print('INFORMACIÓN INICIAL DEL DATASET')
print('=' * 60)

print(f'\n1. DIMENSIONES: {data.shape[0]} filas × {data.shape[1]} columnas')
print(f'\n2. TIPOS DE DATOS:\n{data.dtypes.value_counts()}')
print(f'\n3. PRIMERAS 5 FILAS:')
data.head()

In [ ]:
print('4. ESTADÍSTICAS DESCRIPTIVAS:')
data.describe()

In [ ]:
print('5. VALORES NULOS:')
null_counts = data.isnull().sum()
null_percent = (null_counts / len(data)) * 100
null_summary = pd.DataFrame({'Valores_Nulos': null_counts, 'Porcentaje': null_percent})
print(null_summary[null_summary['Valores_Nulos'] > 0])

print('\n6. DISTRIBUCIÓN DE diagnosis:')
diagnosis_counts = data['diagnosis'].value_counts()
diagnosis_percent = data['diagnosis'].value_counts(normalize=True) * 100
pd.DataFrame({'Conteo': diagnosis_counts, 'Porcentaje': diagnosis_percent})

## 2. Preprocesamiento de Datos

In [ ]:
print('=' * 60)
print('PREPROCESAMIENTO DE DATOS')
print('=' * 60)

# Eliminar columnas no relevantes
columns_to_drop = ['id', 'Unnamed: 32']
data_clean = data.drop(columns=columns_to_drop, errors='ignore')
print(f'\nColumnas eliminadas: {columns_to_drop}')
print(f'Nuevo shape: {data_clean.shape}')

# Separar X e y
X = data_clean.drop('diagnosis', axis=1)
y = data_clean['diagnosis'].map({'B': 0, 'M': 1})
print(f'\nX: {X.shape} | y: distribución {dict(zip(["Benigno(0)", "Maligno(1)"], np.bincount(y)))}')

# Imputación de valores nulos
if X.isnull().sum().sum() > 0:
    imputer = SimpleImputer(strategy='median')
    X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    print(f'Valores nulos imputados: {X.isnull().sum().sum()}')
else:
    X_imputed = X.copy()
    print('✓ No hay valores nulos')

data_processed = pd.concat([X_imputed, y.rename('diagnosis').reset_index(drop=True)], axis=1)
print('\n✓ Preprocesamiento completado')

## 3. Análisis Exploratorio de Datos (EDA)

In [ ]:
print('=' * 60)
print('ANÁLISIS EXPLORATORIO DE DATOS (EDA)')
print('=' * 60)

# Pairplot
print('\n1. PAIRPLOT (coloreado por diagnóstico)...')
sns.set_theme(style='ticks')
sns.pairplot(data_processed, hue='diagnosis')
plt.show()

In [ ]:
# Matriz de correlación
print('\n2. MATRIZ DE CORRELACIÓN...')
corr_matrix = X_imputed.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(20, 16))
sns.heatmap(corr_matrix, mask=mask, vmax=1.0, vmin=-1.0, cmap='viridis', center=0,
            annot=True, annot_kws={'size': 7}, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
plt.title('Matriz de Correlación de Características', fontsize=16, pad=20)
plt.tight_layout()
plt.show()

# Correlaciones fuertes
print('\nCORRELACIONES FUERTES (|r| > 0.9):')
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.9:
            print(f'  {corr_matrix.columns[i]} ↔ {corr_matrix.columns[j]}: r = {corr_matrix.iloc[i, j]:.3f}')

In [ ]:
# Boxplots por diagnóstico
print('\n3. BOXPLOTS POR DIAGNOSIS (10 características con mayor varianza)...')
top_features = X_imputed.var().sort_values(ascending=False).head(10).index.tolist()

fig, axes = plt.subplots(5, 2, figsize=(15, 20))
axes = axes.ravel()

for idx, feature in enumerate(top_features):
    plot_data = pd.DataFrame({
        'Valor': X_imputed[feature],
        'Diagnosis': y.map({0: 'Benigno (B)', 1: 'Maligno (M)'})
    })
    sns.boxplot(x='Diagnosis', y='Valor', data=plot_data, ax=axes[idx],
                palette=['lightgreen', 'lightcoral'])

    benign_vals = X_imputed[y == 0][feature]
    malign_vals = X_imputed[y == 1][feature]
    axes[idx].set_title(f'{feature}\n', fontsize=12)
    axes[idx].text(0.5, 0.95, f'B: μ={benign_vals.mean():.2f}, σ={benign_vals.std():.2f}',
                   transform=axes[idx].transAxes, ha='center', fontsize=9)
    axes[idx].text(0.5, 0.90, f'M: μ={malign_vals.mean():.2f}, σ={malign_vals.std():.2f}',
                   transform=axes[idx].transAxes, ha='center', fontsize=9)

    t_stat, p_val = stats.ttest_ind(benign_vals, malign_vals, equal_var=False)
    stars = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    axes[idx].text(0.5, 0.85, f'p={p_val:.2e} {stars}',
                   transform=axes[idx].transAxes, ha='center', fontsize=9,
                   color='red' if p_val < 0.05 else 'black')

plt.suptitle('Distribución de Características por Diagnóstico', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Histogramas y pruebas de normalidad
print('\n4. HISTOGRAMAS Y PRUEBAS DE NORMALIDAD (KS)...')
test_features = ['radius_mean', 'texture_mean', 'perimeter_mean',
                 'area_mean', 'smoothness_mean', 'compactness_mean']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, feature in enumerate(test_features):
    sns.histplot(data=X_imputed, x=feature, kde=True, ax=axes[idx], bins=30)
    stat, p_value = stats.kstest(stats.zscore(X_imputed[feature].dropna()), 'norm')
    axes[idx].set_title(f'{feature}\nKS p-value = {p_value:.3e}', fontsize=11)
    axes[idx].set_xlabel('')
    normal_text = 'Normal' if p_value > 0.05 else 'No normal'
    axes[idx].text(0.05, 0.95, normal_text, transform=axes[idx].transAxes,
                   fontsize=10, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Distribuciones y Pruebas de Normalidad', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print('\n✓ Análisis exploratorio completado')

## 4. Análisis de Componentes Principales (PCA)

In [ ]:
print('=' * 60)
print('ANÁLISIS DE COMPONENTES PRINCIPALES (PCA)')
print('=' * 60)

# Normalización
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)
X_scaled_df = pd.DataFrame(X_scaled, columns=X_imputed.columns)
print(f'Datos normalizados: {X_scaled.shape} | Media≈{X_scaled.mean():.2f} | Std≈{X_scaled.std():.2f}')

In [ ]:
# PCA exploratorio
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)
explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

# Gráficos de varianza
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.bar(range(1, len(explained_variance) + 1), explained_variance, alpha=0.6)
ax1.plot(range(1, len(explained_variance) + 1), explained_variance, 'r-', marker='o')
ax1.set_xlabel('Componente Principal')
ax1.set_ylabel('Varianza Explicada')
ax1.set_title('Varianza Explicada por Componente')
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, 'b-', marker='o', linewidth=2)
ax2.axhline(y=0.95, color='r', linestyle='--', alpha=0.7, label='95% de varianza')
ax2.axhline(y=0.90, color='g', linestyle='--', alpha=0.7, label='90% de varianza')
ax2.set_xlabel('Número de Componentes')
ax2.set_ylabel('Varianza Acumulada')
ax2.set_title('Varianza Acumulada')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

for pct in [0.80, 0.90, 0.95, 0.99]:
    n = np.argmax(cumulative_variance >= pct) + 1
    print(f'  Componentes para {pct:.0%} de varianza: {n}')

In [ ]:
# PCA óptimo (17 componentes → 99% varianza)
n_components_optimal = 17
pca_optimal = PCA(n_components=n_components_optimal)
X_pca_optimal = pca_optimal.fit_transform(X_scaled)

print(f'PCA con {n_components_optimal} componentes:')
print(f'  Varianza explicada : {pca_optimal.explained_variance_ratio_.sum():.3%}')
print(f'  Shape transformado : {X_pca_optimal.shape}')
print(f'  Tasa de compresión : {(1 - n_components_optimal / X_imputed.shape[1]):.1%}')

# Visualización 2D con elipses
from matplotlib.patches import Ellipse

plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_pca_optimal[:, 0], X_pca_optimal[:, 1],
                     c=y, cmap='coolwarm', alpha=0.7, s=50, edgecolors='k')
plt.xlabel(f"PC1 ({pca_optimal.explained_variance_ratio_[0]:.1%} varianza)")
plt.ylabel(f"PC2 ({pca_optimal.explained_variance_ratio_[1]:.1%} varianza)")
plt.title('Primeras Dos Componentes Principales')
plt.colorbar(scatter, label='Diagnóstico (0=Benigno, 1=Maligno)')
plt.grid(True, alpha=0.3)

for diagnosis_value in [0, 1]:
    mask = y == diagnosis_value
    if np.sum(mask) > 1:
        cov = np.cov(X_pca_optimal[mask, 0], X_pca_optimal[mask, 1])
        lambda_, v = np.linalg.eig(cov)
        lambda_ = np.sqrt(lambda_)
        ell = Ellipse(xy=(np.mean(X_pca_optimal[mask, 0]), np.mean(X_pca_optimal[mask, 1])),
                     width=lambda_[0]*4, height=lambda_[1]*4,
                     angle=np.degrees(np.arctan2(v[1, 0], v[0, 0])),
                     alpha=0.2, color='red' if diagnosis_value == 1 else 'green')
        plt.gca().add_patch(ell)

plt.tight_layout()
plt.show()

# Cargas
loadings = pd.DataFrame(
    pca_optimal.components_.T,
    columns=[f'PC{i+1}' for i in range(n_components_optimal)],
    index=X_imputed.columns
)
print('\nTOP 10 variables — PC1:')
print(loadings['PC1'].abs().sort_values(ascending=False).head(10))
print('\nTOP 10 variables — PC2:')
print(loadings['PC2'].abs().sort_values(ascending=False).head(10))

# Guardar para modelos
X_pca_df = pd.DataFrame(
    X_pca_optimal,
    columns=[f'PC{i+1}' for i in range(n_components_optimal)]
)
print(f'\n✓ PCA completado. X_pca_df shape: {X_pca_df.shape}')

## 5–6. Preparación y División de Datos para Clasificación

In [ ]:
# Variable objetivo y predictores
y_clas = data_clean['diagnosis'].map({'B': 0, 'M': 1}).values
X_pca_clas = X_pca_df.copy()

print(f'Distribución de clases:')
print(f'  Benigno (0): {np.sum(y_clas == 0)} ({np.mean(y_clas == 0):.2%})')
print(f'  Maligno (1): {np.sum(y_clas == 1)} ({np.mean(y_clas == 1):.2%})')

# División estratificada
X_train_clas, X_test_clas, y_train_clas, y_test_clas = train_test_split(
    X_pca_clas, y_clas, test_size=0.2, random_state=42, stratify=y_clas, shuffle=True
)

print(f'\nEntrenamiento: {X_train_clas.shape} | Prueba: {X_test_clas.shape}')

# Separabilidad en espacio PCA
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (c1, c2, title) in zip(axes, [(0, 1, 'PC1 vs PC2'), (2, 3, 'PC3 vs PC4')]):
    for i, (label, color, marker) in enumerate(zip(
        ['Benigno', 'Maligno'], ['lightgreen', 'lightcoral'], ['o', 's']
    )):
        mask = y_clas == i
        ax.scatter(X_pca_clas.iloc[mask, c1], X_pca_clas.iloc[mask, c2],
                   c=color, label=label, alpha=0.7, s=50, marker=marker)
    ax.set_xlabel(f'Componente Principal {c1+1}')
    ax.set_ylabel(f'Componente Principal {c2+1}')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7–9. Modelos de Clasificación

In [ ]:
def evaluar_modelo_clasificacion(modelo, nombre_modelo, X_train, X_test, y_train, y_test):
    """Entrena y evalúa un modelo. Devuelve métricas y modelo entrenado."""
    modelo.fit(X_train, y_train)
    y_pred_train = modelo.predict(X_train)
    y_pred_test = modelo.predict(X_test)
    y_pred_proba = modelo.predict_proba(X_test)[:, 1] if hasattr(modelo, 'predict_proba') else None

    metrics = {
        'Nombre': nombre_modelo,
        'Accuracy Train': accuracy_score(y_train, y_pred_train),
        'Accuracy Test':  accuracy_score(y_test, y_pred_test),
        'Precision Test': precision_score(y_test, y_pred_test),
        'Recall Test':    recall_score(y_test, y_pred_test),
        'F1 Test':        f1_score(y_test, y_pred_test),
        'AUC Test':       roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else np.nan
    }

    cm = confusion_matrix(y_test, y_pred_test)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
    axes[0].set_xlabel('Predicho'); axes[0].set_ylabel('Real')
    axes[0].set_title(f'{nombre_modelo} — Matriz de Confusión')
    report = classification_report(y_test, y_pred_test, output_dict=False)
    axes[1].text(0.1, 0.9, report, fontfamily='monospace', fontsize=10,
                 verticalalignment='top', transform=axes[1].transAxes)
    axes[1].axis('off'); axes[1].set_title(f'{nombre_modelo} — Reporte')
    plt.tight_layout(); plt.show()

    return metrics, modelo

print('✓ Función de evaluación definida')

In [ ]:
print('=' * 60)
print('ENTRENAMIENTO DE MODELOS DE CLASIFICACIÓN')
print('=' * 60)

resultados_clasificacion = []

# Regresión Logística
print('\n1. REGRESIÓN LOGÍSTICA...')
lr_clas = LogisticRegression(random_state=42, max_iter=1000)
metrics_lr, model_lr = evaluar_modelo_clasificacion(
    lr_clas, 'Regresión Logística', X_train_clas, X_test_clas, y_train_clas, y_test_clas)
resultados_clasificacion.append(metrics_lr)

# KNN
print('\n2. K-NEAREST NEIGHBORS (KNN)...')
knn_clas = KNeighborsClassifier(n_neighbors=5)
metrics_knn, model_knn = evaluar_modelo_clasificacion(
    knn_clas, 'KNN (k=5)', X_train_clas, X_test_clas, y_train_clas, y_test_clas)
resultados_clasificacion.append(metrics_knn)

# SVM
print('\n3. SUPPORT VECTOR MACHINE (SVM)...')
svm_clas = SVC(kernel='rbf', probability=True, random_state=42)
metrics_svm, model_svm = evaluar_modelo_clasificacion(
    svm_clas, 'SVM (RBF)', X_train_clas, X_test_clas, y_train_clas, y_test_clas)
resultados_clasificacion.append(metrics_svm)

# Árbol de Decisión
print('\n4. ÁRBOL DE DECISIÓN...')
dt_clas = DecisionTreeClassifier(random_state=42, max_depth=5)
metrics_dt, model_dt = evaluar_modelo_clasificacion(
    dt_clas, 'Árbol de Decisión', X_train_clas, X_test_clas, y_train_clas, y_test_clas)
resultados_clasificacion.append(metrics_dt)

# Random Forest
print('\n5. RANDOM FOREST...')
rf_clas = RandomForestClassifier(random_state=42, n_estimators=100)
metrics_rf, model_rf = evaluar_modelo_clasificacion(
    rf_clas, 'Random Forest', X_train_clas, X_test_clas, y_train_clas, y_test_clas)
resultados_clasificacion.append(metrics_rf)

print('\n✓ Todos los modelos entrenados y evaluados')

## 9. Comparativa Final de Modelos

In [ ]:
from sklearn.metrics import roc_curve as _roc_curve

resultados_df = pd.DataFrame(resultados_clasificacion).sort_values('Accuracy Test', ascending=False)
print('TABLA COMPARATIVA DE MODELOS:')
print(resultados_df.to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
x_pos = np.arange(len(resultados_df)); width = 0.35

axes[0, 0].bar(x_pos - width/2, resultados_df['Accuracy Train'], width, label='Train', alpha=0.8, color='skyblue')
axes[0, 0].bar(x_pos + width/2, resultados_df['Accuracy Test'], width, label='Test', alpha=0.8, color='lightcoral')
axes[0, 0].set_xticks(x_pos); axes[0, 0].set_xticklabels(resultados_df['Nombre'], rotation=45, ha='right')
axes[0, 0].set_title('Comparación de Accuracy'); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

for i, metric in enumerate(['Precision Test', 'Recall Test', 'F1 Test']):
    axes[0, 1].bar(x_pos + i*0.25, resultados_df[metric], width=0.25, label=metric.replace(' Test', ''))
axes[0, 1].set_xticks(x_pos + 0.25); axes[0, 1].set_xticklabels(resultados_df['Nombre'], rotation=45, ha='right')
axes[0, 1].set_title('Métricas (Test)'); axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot([0, 1], [0, 1], 'k--', label='Aleatorio')
colors = ['blue', 'green', 'red', 'purple', 'orange']
models_list = [model_lr, model_knn, model_svm, model_dt, model_rf]
for model, name, color in zip(models_list, resultados_df['Nombre'], colors):
    if hasattr(model, 'predict_proba'):
        fpr, tpr, _ = _roc_curve(y_test_clas, model.predict_proba(X_test_clas)[:, 1])
        auc_score = roc_auc_score(y_test_clas, model.predict_proba(X_test_clas)[:, 1])
        axes[1, 0].plot(fpr, tpr, label=f'{name} (AUC={auc_score:.3f})', color=color)
axes[1, 0].set_title('Curvas ROC Comparativas'); axes[1, 0].legend(loc='lower right'); axes[1, 0].grid(True, alpha=0.3)

best = resultados_df.iloc[0]
info = (f'RESUMEN PCA + CLASIFICACIÓN\n\n'
        f'Componentes PCA: {X_pca_clas.shape[1]}\n'
        f'Varianza expl.  : {pca_optimal.explained_variance_ratio_.sum():.1%}\n'
        f'Mejor modelo    : {best["Nombre"]}\n'
        f'Accuracy (test) : {best["Accuracy Test"]:.1%}\n'
        f'Mejor F1        : {resultados_df["F1 Test"].max():.3f}')
axes[1, 1].axis('off')
axes[1, 1].text(0.1, 0.9, info, fontsize=12, verticalalignment='top',
                transform=axes[1, 1].transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout(); plt.show()

## 10–12. Optimización Bayesiana

In [ ]:
# Modelo base para comparación
y_train_pred_base = model_lr.predict(X_train_clas)
y_test_pred_base = model_lr.predict(X_test_clas)
y_test_proba_base = model_lr.predict_proba(X_test_clas)[:, 1]

base_metrics = {
    'Accuracy Train': accuracy_score(y_train_clas, y_train_pred_base),
    'Accuracy Test':  accuracy_score(y_test_clas, y_test_pred_base),
    'Precision Test': precision_score(y_test_clas, y_test_pred_base),
    'Recall Test':    recall_score(y_test_clas, y_test_pred_base),
    'F1 Test':        f1_score(y_test_clas, y_test_pred_base),
    'AUC Test':       roc_auc_score(y_test_clas, y_test_proba_base)
}
print('MÉTRICAS DEL MODELO BASE (Regresión Logística):')
for k, v in base_metrics.items():
    print(f'  {k}: {v:.3%}')

In [ ]:
import time

try:
    from skopt import BayesSearchCV
    from skopt.space import Real, Categorical, Integer
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'scikit-optimize', '-q'])
    from skopt import BayesSearchCV
    from skopt.space import Real, Categorical, Integer

search_spaces = {
    'C': Real(0.001, 100, prior='log-uniform'),
    'penalty': Categorical(['l1', 'l2']),
    'solver': Categorical(['liblinear', 'saga']),
    'max_iter': Integer(100, 2000),
    'class_weight': Categorical(['balanced', None]),
    'fit_intercept': Categorical([True, False])
}

bayes_search = BayesSearchCV(
    estimator=LogisticRegression(random_state=42),
    search_spaces=search_spaces,
    n_iter=50, cv=5, scoring='f1',
    n_jobs=-1, random_state=42, verbose=1, return_train_score=True
)

t0 = time.time()
bayes_search.fit(X_train_clas, y_train_clas)
print(f'\nTiempo: {time.time()-t0:.1f}s | Mejor F1 (CV): {bayes_search.best_score_:.4f}')
print('Mejores hiperparámetros:')
for k, v in bayes_search.best_params_.items():
    print(f'  {k}: {v}')

optimized_model = bayes_search.best_estimator_
optimized_model.fit(X_train_clas, y_train_clas)

y_train_opt_pred = optimized_model.predict(X_train_clas)
y_test_opt_pred  = optimized_model.predict(X_test_clas)
y_test_opt_proba = optimized_model.predict_proba(X_test_clas)[:, 1]

opt_metrics = {
    'Accuracy Train': accuracy_score(y_train_clas, y_train_opt_pred),
    'Accuracy Test':  accuracy_score(y_test_clas, y_test_opt_pred),
    'Precision Test': precision_score(y_test_clas, y_test_opt_pred),
    'Recall Test':    recall_score(y_test_clas, y_test_opt_pred),
    'F1 Test':        f1_score(y_test_clas, y_test_opt_pred),
    'AUC Test':       roc_auc_score(y_test_clas, y_test_opt_proba)
}
print('\nMÉTRICAS DEL MODELO OPTIMIZADO:')
for k, v in opt_metrics.items():
    print(f'  {k}: {v:.3%}')

## 13. Validación Cruzada del Modelo Optimizado

In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scoring_metrics = {'accuracy': 'accuracy', 'precision': 'precision',
                   'recall': 'recall', 'f1': 'f1', 'roc_auc': 'roc_auc'}

cv_results = {}
for name, scoring in scoring_metrics.items():
    scores = cross_val_score(optimized_model, X_train_clas, y_train_clas,
                             cv=cv, scoring=scoring, n_jobs=-1)
    cv_results[name] = scores
    print(f'  {name.upper():12s}: {scores.mean():.3%} ± {scores.std():.3%}')

# Visualización
cv_data = [{'Métrica': m, 'Score': s} for m, scores in cv_results.items() for s in scores]
cv_df = pd.DataFrame(cv_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x='Métrica', y='Score', data=cv_df, ax=axes[0])
sns.stripplot(x='Métrica', y='Score', data=cv_df, color='black', alpha=0.5, ax=axes[0])
axes[0].set_title('Distribución de Métricas en Validación Cruzada')
axes[0].tick_params(axis='x', rotation=45); axes[0].grid(True, alpha=0.3, axis='y')

folds = np.arange(1, 11)
for metric in ['accuracy', 'f1', 'roc_auc']:
    axes[1].plot(folds, cv_results[metric], marker='o', label=metric, linewidth=2)
axes[1].set_xlabel('Fold'); axes[1].set_ylabel('Puntuación')
axes[1].set_title('Evolución por Fold'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print('\n✓ Validación cruzada completada')

## 14. Sistema Predictivo Final

In [ ]:
import joblib, json
from datetime import datetime

def predict_breast_cancer(features, model, threshold=0.5):
    features = np.array(features).reshape(1, -1)
    proba = model.predict_proba(features)[0]
    label = 'Maligno' if proba[1] >= threshold else 'Benigno'
    confidence = proba[1] if label == 'Maligno' else proba[0]
    return {
        'prediccion': label,
        'probabilidad_maligno': float(proba[1]),
        'probabilidad_benigno': float(proba[0]),
        'confianza': float(confidence),
        'umbral_utilizado': threshold,
        'recomendacion': 'CONSULTAR CON ESPECIALISTA' if label == 'Maligno' else 'SEGUIMIENTO RUTINARIO'
    }

# Ejemplo de predicción
sample_idx = np.random.randint(0, len(X_test_clas))
sample_features = X_test_clas.iloc[sample_idx].values
true_label = 'Maligno' if y_test_clas[sample_idx] == 1 else 'Benigno'

result = predict_breast_cancer(sample_features, optimized_model)
print(f'Muestra #{sample_idx}:')
for k, v in result.items():
    print(f'  {k}: {v}')
print(f'  Diagnóstico real: {true_label}')
print(f'  Correcto: {"✓" if result["prediccion"] == true_label else "✗"}')

# Guardar modelo
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
joblib.dump(optimized_model, f'models/breast_cancer_model_{ts}.pkl')
print(f'\n✓ Modelo guardado: models/breast_cancer_model_{ts}.pkl')

## 15. Red Neuronal Profunda

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight

print(f'TensorFlow {tf.__version__} | GPU: {len(tf.config.list_physical_devices("GPU")) > 0}')

X_train_nn = X_train_clas.values if hasattr(X_train_clas, 'values') else X_train_clas
X_test_nn  = X_test_clas.values  if hasattr(X_test_clas,  'values') else X_test_clas

class_weights = compute_class_weight('balanced', classes=np.unique(y_train_clas), y=y_train_clas)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print(f'Pesos de clases: {class_weight_dict}')

In [ ]:
def build_neural_network(input_dim, dropout_rate=0.3, l2_reg=0.001):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation='relu',
                     kernel_regularizer=regularizers.l2(l2_reg),
                     kernel_initializer='he_normal'),
        layers.BatchNormalization(), layers.Dropout(dropout_rate),
        layers.Dense(32, activation='relu',
                     kernel_regularizer=regularizers.l2(l2_reg),
                     kernel_initializer='he_normal'),
        layers.BatchNormalization(), layers.Dropout(dropout_rate * 0.8),
        layers.Dense(16, activation='relu',
                     kernel_regularizer=regularizers.l2(l2_reg * 0.5),
                     kernel_initializer='he_normal'),
        layers.BatchNormalization(), layers.Dropout(dropout_rate * 0.6),
        layers.Dense(1, activation='sigmoid', kernel_initializer='glorot_uniform')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall'),
                 keras.metrics.AUC(name='auc')]
    )
    return model

nn_model = build_neural_network(X_train_nn.shape[1])
nn_model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1),
    ModelCheckpoint('models/best_neural_model.h5', monitor='val_auc',
                    save_best_only=True, mode='max', verbose=1)
]

history = nn_model.fit(
    X_train_nn, y_train_clas,
    validation_data=(X_test_nn, y_test_clas),
    epochs=100, batch_size=16,
    class_weight=class_weight_dict,
    callbacks=callbacks, verbose=1
)

In [ ]:
best_nn = keras.models.load_model('models/best_neural_model.h5')
test_results = best_nn.evaluate(X_test_nn, y_test_clas, verbose=0)
test_preds   = best_nn.predict(X_test_nn)
test_pred_bin = (test_preds > 0.5).astype(int)

print(f'Loss: {test_results[0]:.4f} | Accuracy: {test_results[1]:.4f} | AUC: {test_results[4]:.4f}')

# Visualización entrenamiento
metrics_to_plot = ['loss', 'accuracy', 'precision', 'recall', 'auc']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()
for idx, metric in enumerate(metrics_to_plot):
    axes[idx].plot(history.history[metric], label='Entrenamiento', lw=2)
    axes[idx].plot(history.history[f'val_{metric}'], label='Validación', lw=2)
    axes[idx].set_title(metric.capitalize()); axes[idx].legend(); axes[idx].grid(True, alpha=0.3)
if 'lr' in history.history:
    axes[5].plot(history.history['lr'], color='purple', lw=2)
    axes[5].set_yscale('log'); axes[5].set_title('Learning Rate'); axes[5].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Curva ROC y matriz de confusión
from sklearn.metrics import roc_curve as _roc_curve, auc as _auc
cm_nn = confusion_matrix(y_test_clas, test_pred_bin)
fpr_nn, tpr_nn, _ = _roc_curve(y_test_clas, test_preds)
roc_auc_nn = _auc(fpr_nn, tpr_nn)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.heatmap(cm_nn, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Matriz de Confusión — Red Neuronal')
axes[0].set_xticklabels(['Benigno', 'Maligno']); axes[0].set_yticklabels(['Benigno', 'Maligno'])

axes[1].plot(fpr_nn, tpr_nn, color='darkorange', lw=2, label=f'AUC = {roc_auc_nn:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--'); axes[1].set_title('Curva ROC')
axes[1].legend(loc='lower right'); axes[1].grid(True, alpha=0.3)

axes[2].hist(test_preds[y_test_clas == 0], alpha=0.7, bins=20, label='Benigno', color='green')
axes[2].hist(test_preds[y_test_clas == 1], alpha=0.7, bins=20, label='Maligno', color='red')
axes[2].axvline(x=0.5, color='black', linestyle='--'); axes[2].legend()
axes[2].set_title('Distribución de Probabilidades')
plt.tight_layout(); plt.show()

print(classification_report(y_test_clas, test_pred_bin, target_names=['Benigno', 'Maligno']))

In [ ]:
# Optimización del umbral
thresholds_test = np.linspace(0.1, 0.9, 50)
f1_scores_thr = [f1_score(y_test_clas, (test_preds > t).astype(int)) for t in thresholds_test]
best_thr_idx = np.argmax(f1_scores_thr)
best_threshold = thresholds_test[best_thr_idx]
best_f1_nn = f1_scores_thr[best_thr_idx]

plt.figure(figsize=(10, 6))
plt.plot(thresholds_test, f1_scores_thr, 'b-', lw=2)
plt.axvline(x=best_threshold, color='r', linestyle='--', label=f'Óptimo: {best_threshold:.3f}')
plt.axvline(x=0.5, color='g', linestyle='--', label='Default: 0.5')
plt.xlabel('Umbral'); plt.ylabel('F1-Score'); plt.title('Optimización del Umbral')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

test_pred_opt = (test_preds > best_threshold).astype(int)
print(f'Umbral óptimo: {best_threshold:.3f} | F1 = {best_f1_nn:.4f}')
print(classification_report(y_test_clas, test_pred_opt, target_names=['Benigno', 'Maligno']))

# Guardar red neuronal
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
best_nn.save(f'models/breast_cancer_neural_{ts}.h5')
print(f'\n✓ Red neuronal guardada: models/breast_cancer_neural_{ts}.h5')
print('\n✓ Pipeline completo finalizado exitosamente')